In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

In [0]:
df_rides = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/teambricks/Bookings.csv")

In [0]:
import re

def clean_columns(df):
    for col_name in df.columns:
        new_col = re.sub(r'[ ,;{}()\n\t=]', '_', col_name)
        df = df.withColumnRenamed(col_name, new_col)
    return df

df_rides = clean_columns(df_rides)

In [0]:
df_rides.write.format("delta") \
    .mode("append") \
    .save("/Volumes/workspace/default/teambricks/bronze/rides")

In [0]:
customers_dim = df_rides.select("Customer_ID", "Customer_Rating")
customers_dim.printSchema()

root
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Rating: string (nullable = true)



In [0]:
vehicle_dim = df_rides.select("Vehicle_Type")
vehicle_dim.printSchema()

root
 |-- Vehicle_Type: string (nullable = true)



In [0]:
pickup = df_rides.select(col("Pickup_Location").alias("Location_Name"))
drop = df_rides.select(col("Drop_Location").alias("Location_Name"))

location_dim = pickup.union(drop)
location_dim.printSchema()

root
 |-- Location_Name: string (nullable = true)



In [0]:
from pyspark.sql.functions import hour, dayofmonth, month, year

time_dim = df_rides.withColumn("Hour", hour("Time")) \
    .withColumn("Day", dayofmonth("Date")) \
    .withColumn("Month", month("Date")) \
    .withColumn("Year", year("Date")) \
    .select("Date", "Time", "Hour", "Day", "Month", "Year")
time_dim.printSchema()

root
 |-- Date: timestamp (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- Hour: integer (nullable = true)
 |-- Day: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Year: integer (nullable = true)



In [0]:
payments_dim = df_rides.select("Payment_Method")
payments_dim.printSchema()

root
 |-- Payment_Method: string (nullable = true)



In [0]:
cancellation_dim = df_rides.select(
    "Canceled_Rides_by_Customer",
    "Canceled_Rides_by_Driver",
    "Incomplete_Rides",
    "Incomplete_Rides_Reason"
)
cancellation_dim.printSchema()

root
 |-- Canceled_Rides_by_Customer: string (nullable = true)
 |-- Canceled_Rides_by_Driver: string (nullable = true)
 |-- Incomplete_Rides: string (nullable = true)
 |-- Incomplete_Rides_Reason: string (nullable = true)



In [0]:
print(cancellation_dim.count())

103024
